# Sampling Distributions
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** a sampling distribution and distinguish it from the population distribution
- **Calculate** the standard error of the mean and explain what it measures
- **Explain** why larger samples produce more reliable estimates of population parameters

> **Tip:** Start by moving the **number of samples slider**, watch the sampling distribution histogram fill in and converge toward a smooth bell curve as more samples are drawn.

---
## How we got here

In *13: Central Limit Theorem* we proved that sample means become normally distributed as n grows. A **sampling distribution** is the name we give to the distribution of any sample statistic, the mean, the proportion, the variance, across all possible samples of the same size. This notebook makes that abstract idea concrete by simulating it.

---
## Why this matters for data science

Every time you report a model's accuracy on a test set, you are reporting a sample statistic, and that statistic has a sampling distribution. Understanding sampling distributions lets you attach uncertainty to any estimate: "our model achieves 84% accuracy" becomes far more useful when paired with "±2 percentage points at 95% confidence." The next notebooks on confidence intervals and hypothesis testing are both built directly on sampling distributions.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots
from tkh_utils import make_int_slider, make_dropdown, make_output, display_widget, register_observer

np.random.seed(42)
_POPULATIONS = {
    "Normal":      np.random.normal(loc=0, scale=1, size=5000),
    "Uniform":     np.random.uniform(low=-np.sqrt(3), high=np.sqrt(3), size=5000),
    "Exponential": np.random.exponential(scale=1, size=5000) - 1,
}

out = make_output()
n_slider          = make_int_slider(5, 200, 5, 30, "Sample size (n):")
num_samples_slider = make_int_slider(10, 2000, 10, 200, "Num samples:")
pop_dropdown      = make_dropdown(["Normal", "Uniform", "Exponential"], "Normal", "Population:")

def render(n, num_samples, pop_type):
    pop       = _POPULATIONS[pop_type]
    mu_pop    = pop.mean()
    sigma_pop = pop.std()
    se        = sigma_pop / np.sqrt(n)

    idx          = np.random.randint(0, len(pop), size=(num_samples, n))
    sample_means = pop[idx].mean(axis=1)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Population Distribution", "Sampling Distribution of x̄"],
    )
    fig.add_trace(go.Histogram(
        x=pop, nbinsx=50,
        marker_color=PALETTE["muted"], opacity=0.75,
        name="Population", showlegend=True,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[mu_pop, mu_pop], y=[0, num_samples // 3],
        mode="lines",
        line=dict(color=PALETTE["primary"], width=2, dash="dash"),
        name=f"μ = {mu_pop:.2f}", showlegend=True,
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=sample_means, nbinsx=40,
        marker_color=PALETTE["primary"], opacity=0.75,
        name="Sample means", showlegend=True,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=[mu_pop, mu_pop], y=[0, num_samples // 5],
        mode="lines",
        line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name=f"Expected mean", showlegend=False,
    ), row=1, col=2)

    layout = base_layout(
        title=f"Sampling Distribution — n = {n}, SE = {se:.3f}  (σ/√n = {sigma_pop:.2f}/√{n})",
        xaxis_title="Value",
        yaxis_title="Count",
    )
    fig.update_layout(layout)
    fig.update_layout(showlegend=True)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer(
    [n_slider, num_samples_slider, pop_dropdown],
    lambda change: render(n_slider.value, num_samples_slider.value, pop_dropdown.value),
)
display_widget([n_slider, num_samples_slider, pop_dropdown], out)
render(n_slider.value, num_samples_slider.value, pop_dropdown.value)


---
## What's happening?

A **sampling distribution** is what you would get if you drew an infinite number of samples of size n from a population and plotted all the resulting sample statistics.

| Concept | Symbol | Definition |
|---------|--------|-----------|
| Population mean | μ | The true average of the entire population |
| Sample mean | x̄ | The average of one sample: varies sample to sample |
| Standard error | SE | Standard deviation of the sampling distribution = σ/√n |
| Sampling distribution | — | Distribution of x̄ across all possible samples |

$$SE_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$$

### The square root law

Doubling the sample size does **not** halve the standard error, it reduces it by a factor of √2 ≈ 1.41. To cut SE in half, you need to **quadruple** n. This is why the first hundred observations are cheap (each one dramatically reduces SE) but the ten-thousandth observation barely moves it.

Go back to the widget and verify this: increase n from 25 to 100 and check that SE dropped by exactly half.

---
## Real-world example: Polling a presidential election

A polling organization wants to estimate the fraction of voters supporting a candidate. They cannot ask everyone, so they sample. Each poll is one draw from the sampling distribution of proportions, and the standard error tells them how much their estimate might vary from the true population proportion.

The chart simulates 500 polls, each of n = 1,000 voters, from a population where 52% support the candidate. Notice:

- **Notice:** The polls cluster tightly around 52%, but vary by roughly ±1.6 percentage points, this is the standard error of a proportion
- **Notice:** Occasionally a poll shows the candidate below 50% (losing), even though they are truly ahead, sampling variability causes this
- **Notice:** The distribution of poll results is approximately normal, confirming the CLT applies to proportions too

> **Discussion question:** A news outlet reports "Candidate leads 52% to 48% with margin of error ±3 points." Based on what you see in this chart, is it possible the race is actually tied? How likely is that outcome?

In [3]:
np.random.seed(77)

# ── Election polling simulation ─────────────────────────────────────────────────
true_p = 0.52       # true population support
n_poll = 1000       # respondents per poll
n_polls = 500       # number of simulated polls

poll_results = np.random.binomial(n_poll, true_p, n_polls) / n_poll
se_prop = np.sqrt(true_p * (1 - true_p) / n_poll)

x_norm = np.linspace(poll_results.min() - 0.01, poll_results.max() + 0.01, 300)
y_norm = stats.norm.pdf(x_norm, true_p, se_prop)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=poll_results * 100, nbinsx=40, histnorm="probability density",
    marker_color=PALETTE["primary"], opacity=0.65, name=f"{n_polls} polls (n={n_poll})",
))
fig.add_trace(go.Scatter(
    x=x_norm * 100, y=y_norm / 100,
    mode="lines", line=dict(color=PALETTE["secondary"], width=2.5),
    name=f"Normal approx  SE={se_prop*100:.2f}%",
))
fig.add_vline(x=true_p * 100, line_dash="dash", line_color=PALETTE["accent"],
              annotation_text=f"True p = {true_p*100:.0f}%")
fig.add_vline(x=50, line_dash="dot", line_color=PALETTE["muted"],
              annotation_text="50% (tie)")
layout = base_layout(
    title=f"Sampling Distribution of Poll Proportions  (true p = {true_p*100:.0f}%, n = {n_poll})",
    xaxis_title="Poll Result (%)",
    yaxis_title="Probability Density",
)
fig.update_layout(layout)
fig.show()

### How standard error depends on sample size

| Sample size n | SE of proportion (p = 0.52) | Width of 95% CI |
|--------------|----------------------------|----------------|
| 100 | 5.0% | ±9.8% |
| 400 | 2.5% | ±4.9% |
| 1,000 | 1.6% | ±3.1% |
| 2,500 | 1.0% | ±2.0% |
| 10,000 | 0.5% | ±1.0% |

---
## Key takeaway

> **The sampling distribution describes how much a statistic varies across repeated samples; the standard error quantifies that variation — and it shrinks only with the square root of sample size.**

---
*Next up: Z-Scores & Standardization — how to place any value on a universal scale so distributions with different units can be compared*